# Full research-session analysis

This tutorial loads the bundled anonymized synthetic export. Change only `EXPORT_PATH` to analyze a dashboard export. The toolbox keeps open-ended payloads nested and exposes stable database entities as joinable pandas tables.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import pandas as pd

from research_toolbox import load_export

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path.cwd().parent
EXPORT_PATH = PROJECT_ROOT / 'examples' / 'full_sessions.synthetic.json'
EXPORT_PATH

## Load and inspect the contract

In [ ]:
data = load_export(EXPORT_PATH)
display(pd.Series(data.metadata, name='value'))
print('Validation warnings:', data.validation_warnings)
pd.DataFrame({'table': data.tables, 'rows': [len(frame) for frame in data.tables.values()]})

## Session and task progress

In [ ]:
overview = data.session_overview()
display(overview[['session_id', 'state', 'group_name', 'condition_id', 'session_duration', 'total_tokens']])
display(data.task_progress()[['session_id', 'task_id', 'task_type', 'status', 'is_complete', 'elapsed_seconds']])

## Conditions and outcome metrics

In [ ]:
condition_metrics = data.condition_summary(metrics=['session_duration', 'total_tokens', 'essay_word_count'])
display(condition_metrics)
plot_data = overview.dropna(subset=['condition_id', 'total_tokens'])
ax = plot_data.plot.bar(x='condition_id', y='total_tokens', legend=False, title='Tokens by assigned condition')
ax.set_ylabel('tokens')
plt.tight_layout()

## Questions, surveys, and essays

In [ ]:
display(data.question_score_summary())
display(data.survey_response_summary())
display(data.essay_summary()[['session_id', 'essay_id', 'word_count', 'calculated_word_count']])

# Definitions join directly to responses; no reference lookup is required.
response_details = data.question_responses.merge(
    data.questions[['question_id', 'prompt', 'question_type']], on='question_id', how='left'
)
display(response_details[['session_id', 'question_id', 'prompt', 'is_answered', 'effective_score']])

## Chats, LLM requests, and telemetry

In [ ]:
display(data.chat_usage_summary())
display(data.llm_requests[['session_id', 'llm_request_id', 'timing_mode', 'status', 'timing_parameters']])
timeline = data.telemetry_timeline('session-complete')
display(timeline[['event_time_dt', 'event_type', 'payload_json']])
display(data.session_timeline('session-complete')[['occurred_at_dt', 'event_kind', 'source_id']].head(10))

## Materialize reusable tables

In [ ]:
with TemporaryDirectory() as directory:
    manifest = data.materialize(
        directory, format='parquet', tables=['sessions', 'session_tasks', 'telemetry_events']
    )
    display(pd.DataFrame(manifest['tables']).T[['file', 'rows', 'json_encoded_columns']])

# Use a durable directory instead of TemporaryDirectory when you want to keep the files.